# Subnetwork uncertainty: entropy of each generated token

Subnetworks are generated as in `generate_subnetwork.py` (netmedgpt.pt on the curated KG, top-5 sampling, T = 1,
200 paths per direction, consensus frequency >= 0.15), for the DrugMechDB pairs of `drugmechdb_evaluation.ipynb`.
In addition, at every generated position the **entropy** of the model's prediction is computed, with the softmax over
the valid tokens only (relation tokens at relation positions, nodes at node positions), normalised to 0-1.

Each triplet (h, r, t) gets the mean entropy of its tail node (and relation) over the paths it was sampled in.
Is the entropy informative? Low-entropy triplets should more often be **KG edges** and more often reach a
**DrugMechDB mechanism node**.

In [ ]:
import os
import sys
if os.path.basename(os.getcwd()) == "revision_scripts":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import json
import math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy import stats
from tqdm import tqdm

from netmedgpt.model import TransformerModel
from config import set_seed

PAIRS = "data/new_analysis/drugmechdb/pairs.csv"     # written by drugmechdb_evaluation.ipynb (disease, drug, mech_nodes)
N_PAIRS = None        # None = all pairs
N_METAPATH = 200      # paths per direction (generate_subnetwork.py default)
TEMPERATURE = 1.0
TOP_K = 5             # sampling among the top-5 tokens, as in generate_subnetwork.py
FREQ_THRESHOLD = 0.15
GPU = 0
SEED = 1
OUT_DIR = "data/new_analysis/subnetwork_uncertainty_entropy"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# ── Curated KG and model (as in generate_subnetwork.py) ───────────────────────
set_seed(SEED)
device = torch.device(f"cuda:{GPU}" if torch.cuda.is_available() else "cpu")
nodes = pd.read_csv("data/KG/nodes_preprocessed.csv")
edge = pd.read_csv("data/KG/edges_preprocessed.csv")
feat = torch.load("data/KG/embeddings_preprocessedKG.pt")

mask_token = int(edge["z_index"].max()) + 1
relation_token = int(edge.loc[edge["relation"] == "indication", "z_index"].iloc[0])
relation_mask_index = pd.concat([edge[["relation", "z_index"]].drop_duplicates(),
                                 pd.DataFrame([["mask", mask_token]], columns=["relation", "z_index"])], ignore_index=True)
entity = list(nodes["node_type"].unique()) + list(edge["relation"].unique()) + ["mask"]

checkpoint = torch.load("data/model_checkpoints/netmedgpt.pt", map_location=device)
param = checkpoint["parameters"]
seq_len = param["walk_length"] * 2 - 1
model = TransformerModel(mask_token + 1, param["hidden_channels"], param["nhead"], param["N_encoder_layers"], seq_len,
                         device=device, feat=feat, nodes=nodes, entity=entity,
                         relation_mask_index=relation_mask_index, pos_emb="fixed").to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# valid tokens: nodes at node positions (even), relations at relation positions (odd)
node_tokens = torch.tensor(nodes["node_index"].values, dtype=torch.long, device=device)
rel_tokens = torch.tensor(sorted(edge["z_index"].unique()), dtype=torch.long, device=device)
kg_pairs = set(zip(edge["x_index"], edge["y_index"])) | set(zip(edge["y_index"], edge["x_index"]))
name = dict(zip(nodes["node_index"], nodes["node_name"]))

In [ ]:
def generate_with_entropy(walk_input, n_metapath, T=1.0):
    """metapath_generation_batched of generate_subnetwork.py, plus the normalised entropy at each generated position
    (softmax over the valid tokens of that position)."""
    walks = walk_input.expand(n_metapath, -1).clone()
    entropy = torch.full(walks.shape, float("nan"), device=walks.device)
    for pos in range(walks.shape[1]):
        if walks[0, pos].item() != mask_token:
            continue
        with torch.inference_mode():
            logits = model(walks)[:, pos, :]
        # sampling exactly as in generate_subnetwork.py
        topk_logits, topk_indices = torch.topk(logits, k=TOP_K, dim=-1)
        sampled = torch.multinomial(F.softmax(topk_logits / T, dim=-1), num_samples=1)
        walks[:, pos] = topk_indices.gather(1, sampled).squeeze(-1)
        # entropy over the valid tokens of this position
        valid = rel_tokens if pos % 2 == 1 else node_tokens
        p = F.softmax(logits[:, valid].float(), dim=-1)
        entropy[:, pos] = -(p * torch.log(p + 1e-12)).sum(-1) / math.log(len(valid))
    return walks.cpu().numpy(), entropy.cpu().numpy()


def triplets(walks, entropy, direction):
    """one row per (path, triplet); the first triplet of a path is the query pair itself"""
    rows = []
    for w, h in zip(walks, entropy):
        for k in range(2, len(w) - 2, 2):
            rows.append((direction, w[k], w[k + 1], w[k + 2], h[k + 1], h[k + 2]))
    return rows

In [ ]:
# ── Generate subnetworks for the DrugMechDB pairs ────────────────────────────
pairs = pd.read_csv(PAIRS)
if N_PAIRS:
    pairs = pairs.sample(N_PAIRS, random_state=SEED)

out = []
for disease, drug, mech in tqdm(zip(pairs["disease"], pairs["drug"], pairs["mech_nodes"].astype(str)), total=len(pairs)):
    mech_nodes = {int(x) for x in mech.split("|") if x}
    rows = []
    for direction, (a, b) in [("forward", (disease, drug)), ("backward", (drug, disease))]:
        walk = torch.tensor([[a, relation_token, b] + [mask_token] * (seq_len - 3)], device=device)
        rows += triplets(*generate_with_entropy(walk, N_METAPATH, TEMPERATURE), direction)
    df = pd.DataFrame(rows, columns=["direction", "h", "r", "t", "relation_entropy", "tail_entropy"])
    # per unique triplet: frequency (share of paths of its direction) and mean entropy
    df = (df.groupby(["direction", "h", "r", "t"])
            .agg(count=("tail_entropy", "size"), tail_entropy=("tail_entropy", "mean"),
                 relation_entropy=("relation_entropy", "mean")).reset_index())
    df["frequency"] = df["count"] / N_METAPATH
    df["disease"], df["drug"] = disease, drug
    df["in_KG"] = [(h, t) in kg_pairs for h, t in zip(df["h"], df["t"])]
    df["tail_is_mechanism_node"] = df["t"].isin(mech_nodes)
    out.append(df)

trip = pd.concat(out, ignore_index=True)
trip["consensus"] = trip["frequency"] >= FREQ_THRESHOLD
trip.to_csv(os.path.join(OUT_DIR, "triplets.csv"), index=False)
print(f"{len(pairs)} pairs, {len(trip)} triplets ({trip['consensus'].sum()} in the consensus subnetworks)")

In [ ]:
# ── Is the entropy informative? ───────────────────────────────────────────────
summary = {}
for subset, g in [("all generated triplets", trip), ("consensus triplets", trip[trip["consensus"]])]:
    q = pd.qcut(g["tail_entropy"], 4, labels=["Q1 (lowest entropy)", "Q2", "Q3", "Q4 (highest entropy)"])
    s = {"triplets": len(g), "median tail entropy": g["tail_entropy"].median()}
    for target in ["in_KG", "tail_is_mechanism_node"]:
        s.update({f"{target} (%) {k}": 100 * v for k, v in g.groupby(q, observed=True)[target].mean().items()})
        s[f"Mann-Whitney p ({target}: lower entropy)"] = stats.mannwhitneyu(
            g.loc[g[target], "tail_entropy"], g.loc[~g[target], "tail_entropy"], alternative="less").pvalue
    s["Spearman(tail entropy, frequency)"] = stats.spearmanr(g["tail_entropy"], g["frequency"])[0]
    summary[subset] = s
summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(OUT_DIR, "summary.csv"))
summary.round(4)

In [ ]:
# example: the consensus subnetwork of one pair with frequency and entropy
d, r = pairs.iloc[0][["disease", "drug"]]
ex = trip[(trip["disease"] == d) & (trip["drug"] == r) & trip["consensus"]].copy()
ex["h"], ex["t"] = ex["h"].map(name), ex["t"].map(name)
ex.sort_values("tail_entropy")[["direction", "h", "t", "frequency", "tail_entropy", "in_KG", "tail_is_mechanism_node"]]